# 02 - Carga de Datos y Mapa Geografico por Comunas
**Proyecto:** Indice de Ingresos Operacionales - Cali  
**Equipo:** ITT Cali Inteligente - Gobierno de Datos  
**Repositorio:** https://github.com/j0rg3c45/Indice_ingresos_operacionales.git

## Objetivo
1. Clonar/actualizar el repositorio desde GitHub (Colab) o usar carpeta local
2. Cargar el archivo Excel del Registro Mercantil 2025
3. Cargar el GeoJSON de comunas de Cali desde `data/info_geo/`
4. Calcular indicadores economicos por comuna
5. Generar mapas coropleticos (estatico e interactivo)

## 1. Instalacion de dependencias

In [ ]:
# Descomentar si faltan paquetes (Colab)
# !pip install pandas openpyxl geopandas folium mapclassify matplotlib seaborn

## 2. Deteccion de entorno y carga del repositorio

In [ ]:
import os
from pathlib import Path

# Configuracion del repositorio
REPO_URL = "https://github.com/j0rg3c45/Indice_ingresos_operacionales.git"
REPO_NAME = "Indice_ingresos_operacionales"

# Detectar entorno: Colab o Local
EN_COLAB = os.path.exists("/content")

if EN_COLAB:
    WORK_DIR = Path("/content") / REPO_NAME
    if not WORK_DIR.exists():
        print(f"Clonando repositorio: {REPO_URL}")
        os.system(f"git clone {REPO_URL}")
    else:
        print("Repositorio ya existe, actualizando...")
        os.system(f"cd {WORK_DIR} && git pull")
else:
    # Local: subir un nivel desde notebooks_py/
    WORK_DIR = Path(os.getcwd()).parent
    if not (WORK_DIR / "README.md").exists():
        WORK_DIR = Path(os.getcwd())

DATA_DIR = WORK_DIR / "data"
GEO_DIR = DATA_DIR / "info_geo" / "geojson_comunas"
OUTPUT_DIR = WORK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Entorno: {'Google Colab' if EN_COLAB else 'Local'}")
print(f"Directorio de trabajo: {WORK_DIR}")
print(f"Directorio de datos: {DATA_DIR}")
print(f"Directorio GeoJSON: {GEO_DIR}")

## 3. Carga del Registro Mercantil 2025

In [ ]:
import pandas as pd
import numpy as np

# Buscar el archivo Excel en data/
archivos_excel = list(DATA_DIR.glob("*.xlsx"))
if archivos_excel:
    ARCHIVO_EXCEL = archivos_excel[0]
else:
    ARCHIVO_EXCEL = DATA_DIR / "Registro mercantil 2025_.xlsx"

print(f"Cargando: {ARCHIVO_EXCEL.name}")
df = pd.read_excel(ARCHIVO_EXCEL, engine="openpyxl")
df.columns = df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)

print(f"Registros: {len(df):,}")
print(f"Columnas: {len(df.columns)}")
print(f"\nColumnas disponibles:")
for i, c in enumerate(df.columns, 1):
    print(f"  {i:2d}. {c}")

## 4. Carga del GeoJSON de comunas

In [ ]:
import geopandas as gpd
import zipfile

# Buscar GeoJSON en data/info_geo/
geojson_path = None

# Opcion 1: archivo ya descomprimido
geojson_files = list((DATA_DIR / "info_geo").glob("**/*.geojson"))
if geojson_files:
    geojson_path = geojson_files[0]
else:
    # Opcion 2: descomprimir ZIP
    zip_files = list((DATA_DIR / "info_geo").glob("*.zip"))
    if zip_files:
        zip_path = zip_files[0]
        extract_dir = DATA_DIR / "info_geo" / zip_path.stem.lower()
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        geojson_files = list(extract_dir.glob("**/*.geojson"))
        if geojson_files:
            geojson_path = geojson_files[0]

if geojson_path:
    print(f"GeoJSON cargado: {geojson_path.name}")
    gdf_comunas = gpd.read_file(geojson_path)
    
    # Normalizar CRS a WGS84
    if gdf_comunas.crs is None:
        gdf_comunas = gdf_comunas.set_crs("EPSG:4326")
    elif gdf_comunas.crs.to_epsg() != 4326:
        gdf_comunas = gdf_comunas.to_crs("EPSG:4326")
    
    print(f"Comunas en GeoJSON: {len(gdf_comunas)}")
    print(f"CRS: {gdf_comunas.crs}")
    print(f"Columnas: {list(gdf_comunas.columns)}")
    print()
    print(gdf_comunas[["comuna", "nombre"]].to_string())
else:
    print("[!] No se encontro GeoJSON. Coloca el archivo en data/info_geo/")
    gdf_comunas = None

## 5. Preparacion de indicadores por comuna

In [ ]:
# Identificar columnas clave
col_comuna = [c for c in df.columns if 'comuna' in c][0]
col_ingresos = [c for c in df.columns if 'ingreso' in c][0]
col_empleo = [c for c in df.columns if 'personal' in c][0]
col_tamano = [c for c in df.columns if 'tama' in c][0]
col_ciiu = [c for c in df.columns if 'codigo' in c and 'ciiu' in c][0]

# Convertir a numerico
df[col_ingresos] = pd.to_numeric(df[col_ingresos], errors='coerce')
df[col_empleo] = pd.to_numeric(df[col_empleo], errors='coerce')

# Filtrar registros con comuna valida
df_con_comuna = df[df[col_comuna].notna()].copy()
print(f'Registros con comuna: {len(df_con_comuna):,} de {len(df):,}')

# Calcular indicadores por comuna
indicadores = df_con_comuna.groupby(col_comuna).agg(
    total_empresas=(col_comuna, 'size'),
    ingresos_promedio=(col_ingresos, 'mean'),
    ingresos_mediana=(col_ingresos, 'median'),
    ingresos_total=(col_ingresos, 'sum'),
    empleo_total=(col_empleo, 'sum'),
    empleo_promedio=(col_empleo, 'mean'),
).reset_index()

# Tasa de microempresas
micro = df_con_comuna[df_con_comuna[col_tamano].str.contains('MICRO', case=False, na=False)]
tasa_micro = micro.groupby(col_comuna).size().reset_index(name='n_micro')
indicadores = indicadores.merge(tasa_micro, on=col_comuna, how='left')
indicadores['pct_micro'] = (indicadores['n_micro'] / indicadores['total_empresas'] * 100).round(1)

# Diversidad economica (CIIU distintos)
diversidad = df_con_comuna.groupby(col_comuna)[col_ciiu].nunique().reset_index(name='n_ciiu_distintos')
indicadores = indicadores.merge(diversidad, on=col_comuna, how='left')

print(f'\nIndicadores calculados para {len(indicadores)} comunas')
print()

# --- TABLA VISUAL DE INDICADORES ---
tabla_display = indicadores.sort_values('total_empresas', ascending=False).copy()
tabla_display['ingresos_prom'] = tabla_display['ingresos_promedio'].apply(
    lambda x: f'${x/1e6:.1f}M' if pd.notna(x) and x >= 1e6 else (f'${x/1e3:.0f}K' if pd.notna(x) and x > 0 else '$0')
)
tabla_display['ingresos_tot'] = tabla_display['ingresos_total'].apply(
    lambda x: f'${x/1e9:.2f}B' if pd.notna(x) and abs(x) >= 1e9 else (f'${x/1e6:.0f}M' if pd.notna(x) else 'N/A')
)
tabla_display['empleo_prom'] = tabla_display['empleo_promedio'].apply(lambda x: f'{x:.1f}' if pd.notna(x) else 'N/A')

cols_tabla = [col_comuna, 'total_empresas', 'ingresos_prom', 'ingresos_tot',
             'empleo_total', 'empleo_prom', 'pct_micro', 'n_ciiu_distintos']
headers = ['Comuna', 'Empresas', 'Ingreso Prom', 'Ingreso Total', 'Empleo', 'Emp/Empresa', '% Micro', 'CIIU']

print('=' * 100)
print('INDICADORES ECONOMICOS POR COMUNA - Registro Mercantil Cali 2025')
print('=' * 100)
print(f'{headers[0]:<12} {headers[1]:>9} {headers[2]:>13} {headers[3]:>14} {headers[4]:>8} {headers[5]:>11} {headers[6]:>7} {headers[7]:>5}')
print('-' * 100)
for _, row in tabla_display.iterrows():
    print(f"{row[col_comuna]:<12} {row['total_empresas']:>9,} {row['ingresos_prom']:>13} {row['ingresos_tot']:>14} {int(row['empleo_total']):>8,} {row['empleo_prom']:>11} {row['pct_micro']:>6.1f}% {int(row['n_ciiu_distintos']):>5}")
print('=' * 100)
print(f"TOTAL: {tabla_display['total_empresas'].sum():,} empresas | {int(tabla_display['empleo_total'].sum()):,} empleos")

## 6. Cruce de datos con GeoJSON

In [ ]:
if gdf_comunas is not None:
    # El GeoJSON tiene columna 'nombre' con formato "Comuna 6", "Comuna 17"
    # El Registro Mercantil tiene formato "Comuna 02", "Comuna 17" (con cero a la izquierda)
    
    # Normalizar: extraer numero y crear clave comun
    gdf_comunas["comuna_num"] = gdf_comunas["comuna"].astype(int)
    gdf_comunas["comuna_key"] = "Comuna " + gdf_comunas["comuna_num"].astype(str).str.zfill(2)
    
    # Verificar formato en datos del registro
    print("Formato en Registro Mercantil:", indicadores[col_comuna].head(5).tolist())
    print("Formato en GeoJSON (key):", gdf_comunas["comuna_key"].head(5).tolist())
    
    # Merge
    gdf_merged = gdf_comunas.merge(
        indicadores,
        left_on="comuna_key",
        right_on=col_comuna,
        how="left"
    )
    
    n_match = gdf_merged["total_empresas"].notna().sum()
    print(f"\nComunas con datos cruzados: {n_match} de {len(gdf_merged)}")
    
    # Calcular densidad empresarial (empresas por hectarea)
    gdf_merged["area_ha"] = gdf_merged["area"] / 10000  # m2 a hectareas
    gdf_merged["densidad_empresarial"] = (gdf_merged["total_empresas"] / gdf_merged["area_ha"]).round(2)
    
    if n_match == 0:
        print("\n[!] No hubo match. Revisar nombres de comuna.")
else:
    gdf_merged = None
    print("[!] No hay GeoJSON cargado.")

## 7. Mapa coropletico - Total de empresas por comuna

In [ ]:
import matplotlib.pyplot as plt

if gdf_merged is not None and gdf_merged["total_empresas"].notna().any():
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    gdf_merged.plot(
        column="total_empresas",
        cmap="YlOrRd",
        linewidth=0.8,
        edgecolor="0.3",
        legend=True,
        legend_kwds={"label": "Total de empresas", "shrink": 0.7},
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "Sin datos"}
    )
    
    # Etiquetas de comuna
    for idx, row in gdf_merged.iterrows():
        centroid = row.geometry.centroid
        label = str(int(row["comuna_num"])) if pd.notna(row.get("comuna_num")) else ""
        ax.annotate(label, xy=(centroid.x, centroid.y), ha="center", fontsize=8, fontweight="bold")
    
    ax.set_title("Densidad Empresarial por Comuna - Cali 2025", fontsize=14, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    
    fig.savefig(OUTPUT_DIR / "mapa_total_empresas_comuna.png", dpi=150, bbox_inches="tight")
    print("[OK] Guardado: outputs/mapa_total_empresas_comuna.png")
    plt.show()
else:
    print("[!] No se puede generar mapa.")

## 8. Mapa coropletico - Ingresos promedio por comuna

In [ ]:
if gdf_merged is not None and gdf_merged["ingresos_promedio"].notna().any():
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    gdf_merged.plot(
        column="ingresos_promedio",
        cmap="Blues",
        linewidth=0.8,
        edgecolor="0.3",
        legend=True,
        legend_kwds={"label": "Ingresos promedio ($)", "shrink": 0.7},
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "Sin datos"}
    )
    
    for idx, row in gdf_merged.iterrows():
        centroid = row.geometry.centroid
        label = str(int(row["comuna_num"])) if pd.notna(row.get("comuna_num")) else ""
        ax.annotate(label, xy=(centroid.x, centroid.y), ha="center", fontsize=8, fontweight="bold")
    
    ax.set_title("Ingresos Operacionales Promedio por Comuna - Cali 2025", fontsize=14, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    
    fig.savefig(OUTPUT_DIR / "mapa_ingresos_promedio_comuna.png", dpi=150, bbox_inches="tight")
    print("[OK] Guardado: outputs/mapa_ingresos_promedio_comuna.png")
    plt.show()
else:
    print("[!] No se puede generar mapa de ingresos.")

## 9. Mapa coropletico - Empleo total por comuna

In [ ]:
if gdf_merged is not None and gdf_merged["empleo_total"].notna().any():
    fig, ax = plt.subplots(1, 1, figsize=(12, 10))
    
    gdf_merged.plot(
        column="empleo_total",
        cmap="Greens",
        linewidth=0.8,
        edgecolor="0.3",
        legend=True,
        legend_kwds={"label": "Empleo total", "shrink": 0.7},
        ax=ax,
        missing_kwds={"color": "lightgrey", "label": "Sin datos"}
    )
    
    for idx, row in gdf_merged.iterrows():
        centroid = row.geometry.centroid
        label = str(int(row["comuna_num"])) if pd.notna(row.get("comuna_num")) else ""
        ax.annotate(label, xy=(centroid.x, centroid.y), ha="center", fontsize=8, fontweight="bold")
    
    ax.set_title("Empleo Total por Comuna - Cali 2025", fontsize=14, fontweight="bold")
    ax.set_axis_off()
    plt.tight_layout()
    
    fig.savefig(OUTPUT_DIR / "mapa_empleo_total_comuna.png", dpi=150, bbox_inches="tight")
    print("[OK] Guardado: outputs/mapa_empleo_total_comuna.png")
    plt.show()
else:
    print("[!] No se puede generar mapa de empleo.")

## 10. Mapa interactivo con Folium

In [ ]:
import folium

if gdf_merged is not None and gdf_merged["total_empresas"].notna().any():
    # Centro de Cali
    centro = [3.4516, -76.5320]
    
    m = folium.Map(location=centro, zoom_start=12, tiles="CartoDB positron")
    
    # Capa coropletica - Total empresas
    folium.Choropleth(
        geo_data=gdf_merged.to_json(),
        data=gdf_merged,
        columns=["comuna_key", "total_empresas"],
        key_on="feature.properties.comuna_key",
        fill_color="YlOrRd",
        fill_opacity=0.7,
        line_opacity=0.5,
        legend_name="Total de empresas por comuna",
        name="Densidad empresarial"
    ).add_to(m)
    
    # Tooltips con info
    style_function = lambda x: {"fillOpacity": 0, "weight": 0}
    
    folium.GeoJson(
        gdf_merged.to_json(),
        name="Info por comuna",
        tooltip=folium.GeoJsonTooltip(
            fields=["comuna_key", "total_empresas", "empleo_total", "pct_micro", "n_ciiu_distintos", "densidad_empresarial"],
            aliases=["Comuna", "Empresas", "Empleo total", "% Micro", "Actividades CIIU", "Empresas/ha"],
            localize=True
        ),
        style_function=style_function
    ).add_to(m)
    
    # Capas base
    folium.TileLayer("OpenStreetMap", name="OpenStreetMap").add_to(m)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
        name="Esri Satelite"
    ).add_to(m)
    
    folium.LayerControl().add_to(m)
    
    # Guardar HTML
    m.save(str(OUTPUT_DIR / "mapa_interactivo_comunas.html"))
    print("[OK] Guardado: outputs/mapa_interactivo_comunas.html")
    
    m
else:
    print("[!] No se puede generar mapa interactivo.")

## 11. Tabla resumen de indicadores por comuna

In [ ]:
if gdf_merged is not None:
    resumen = gdf_merged[[
        'comuna_key', 'total_empresas', 'ingresos_promedio', 'ingresos_total',
        'empleo_total', 'pct_micro', 'n_ciiu_distintos', 'densidad_empresarial', 'area_ha'
    ]].copy()
    
    resumen = resumen.sort_values('total_empresas', ascending=False)
    
    # Formatear valores
    resumen['ing_prom'] = resumen['ingresos_promedio'].apply(
        lambda x: f'${x/1e6:.1f}M' if pd.notna(x) and x >= 1e6 else (f'${x/1e3:.0f}K' if pd.notna(x) and x > 0 else '$0')
    )
    resumen['ing_total'] = resumen['ingresos_total'].apply(
        lambda x: f'${x/1e9:.2f}B' if pd.notna(x) and abs(x) >= 1e9 else (f'${x/1e6:.0f}M' if pd.notna(x) else 'N/A')
    )
    resumen['dens'] = resumen['densidad_empresarial'].apply(lambda x: f'{x:.1f}' if pd.notna(x) else 'N/A')
    resumen['area_fmt'] = resumen['area_ha'].apply(lambda x: f'{x:.0f} ha' if pd.notna(x) else 'N/A')
    
    print('=' * 115)
    print('TABLA RESUMEN - INDICADORES POR COMUNA CON GEOJSON (22 comunas mapeadas)')
    print('=' * 115)
    print(f'{"Comuna":<12} {"Empresas":>9} {"Ing.Prom":>10} {"Ing.Total":>12} {"Empleo":>8} {"% Micro":>8} {"CIIU":>5} {"Emp/ha":>7} {"Area":>9}')
    print('-' * 115)
    for _, row in resumen.iterrows():
        comuna = row['comuna_key'] if pd.notna(row['comuna_key']) else 'N/A'
        empresas = f"{int(row['total_empresas']):,}" if pd.notna(row['total_empresas']) else 'N/A'
        empleo = f"{int(row['empleo_total']):,}" if pd.notna(row['empleo_total']) else 'N/A'
        pct_m = f"{row['pct_micro']:.1f}%" if pd.notna(row['pct_micro']) else 'N/A'
        ciiu = f"{int(row['n_ciiu_distintos'])}" if pd.notna(row['n_ciiu_distintos']) else 'N/A'
        print(f"{comuna:<12} {empresas:>9} {row['ing_prom']:>10} {row['ing_total']:>12} {empleo:>8} {pct_m:>8} {ciiu:>5} {row['dens']:>7} {row['area_fmt']:>9}")
    print('=' * 115)
    
    # Totales
    tot_emp = resumen['total_empresas'].sum()
    tot_empleo = resumen['empleo_total'].sum()
    tot_ing = resumen['ingresos_total'].sum()
    print(f"\nTOTALES (22 comunas mapeadas):")
    print(f"  Empresas: {int(tot_emp):,}")
    print(f"  Empleo:   {int(tot_empleo):,}")
    print(f"  Ingresos: ${tot_ing/1e12:.2f}T" if tot_ing >= 1e12 else f"  Ingresos: ${tot_ing/1e9:.2f}B")
else:
    print('[!] No hay datos cruzados disponibles.')

## 12. Generar reporte consolidado.txt

In [ ]:
from datetime import datetime

REPORTE_PATH = OUTPUT_DIR / 'consolidado.txt'

with open(REPORTE_PATH, 'w', encoding='utf-8') as f:
    f.write('=' * 100 + '\n')
    f.write('REPORTE CONSOLIDADO - INDICE DE INGRESOS OPERACIONALES\n')
    f.write('Registro Mercantil 2025 - Santiago de Cali\n')
    f.write(f'Fecha de generacion: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n')
    f.write('=' * 100 + '\n\n')
    
    # 1. Info general
    f.write('-' * 100 + '\n')
    f.write('1. INFORMACION GENERAL DEL DATASET\n')
    f.write('-' * 100 + '\n')
    f.write(f'  Archivo fuente: Registro mercantil 2025_.xlsx\n')
    f.write(f'  Total registros: {len(df):,}\n')
    f.write(f'  Total columnas: {len(df.columns)}\n')
    f.write(f'  Registros con comuna: {len(df_con_comuna):,}\n')
    f.write(f'  Comunas en datos: {df_con_comuna[col_comuna].nunique()}\n')
    f.write(f'  Comunas en GeoJSON: {len(gdf_comunas) if gdf_comunas is not None else 0}\n')
    f.write(f'  Comunas cruzadas: {gdf_merged["total_empresas"].notna().sum() if gdf_merged is not None else 0}\n\n')
    
    # 2. Indicadores por comuna (todas)
    f.write('-' * 100 + '\n')
    f.write('2. INDICADORES ECONOMICOS POR COMUNA\n')
    f.write('-' * 100 + '\n')
    f.write(f'{"Comuna":<12} {"Empresas":>9} {"Ing.Promedio":>14} {"Ing.Total":>14} {"Empleo":>8} {"Emp/Emp":>8} {"% Micro":>8} {"CIIU":>5}\n')
    f.write('-' * 100 + '\n')
    
    ind_sorted = indicadores.sort_values('total_empresas', ascending=False)
    for _, row in ind_sorted.iterrows():
        ing_p = f'${row["ingresos_promedio"]/1e6:.1f}M' if pd.notna(row['ingresos_promedio']) and row['ingresos_promedio'] >= 1e6 else (f'${row["ingresos_promedio"]/1e3:.0f}K' if pd.notna(row['ingresos_promedio']) and row['ingresos_promedio'] > 0 else '$0')
        ing_t = f'${row["ingresos_total"]/1e9:.2f}B' if pd.notna(row['ingresos_total']) and abs(row['ingresos_total']) >= 1e9 else (f'${row["ingresos_total"]/1e6:.0f}M' if pd.notna(row['ingresos_total']) else 'N/A')
        emp_p = f'{row["empleo_promedio"]:.1f}' if pd.notna(row['empleo_promedio']) else 'N/A'
        f.write(f"{row[col_comuna]:<12} {int(row['total_empresas']):>9,} {ing_p:>14} {ing_t:>14} {int(row['empleo_total']):>8,} {emp_p:>8} {row['pct_micro']:>7.1f}% {int(row['n_ciiu_distintos']):>5}\n")
    
    f.write('-' * 100 + '\n')
    f.write(f"TOTAL: {ind_sorted['total_empresas'].sum():,.0f} empresas | {ind_sorted['empleo_total'].sum():,.0f} empleos\n\n")
    
    # 3. Resumen con GeoJSON
    if gdf_merged is not None:
        f.write('-' * 100 + '\n')
        f.write('3. INDICADORES CON CRUCE GEOGRAFICO (22 comunas mapeadas)\n')
        f.write('-' * 100 + '\n')
        f.write(f'{"Comuna":<12} {"Empresas":>9} {"Ing.Prom":>10} {"Ing.Total":>12} {"Empleo":>8} {"% Micro":>8} {"CIIU":>5} {"Emp/ha":>7} {"Area(ha)":>9}\n')
        f.write('-' * 100 + '\n')
        
        geo_sorted = gdf_merged.sort_values('total_empresas', ascending=False)
        for _, row in geo_sorted.iterrows():
            if pd.isna(row.get('total_empresas')):
                continue
            comuna = row['comuna_key'] if pd.notna(row.get('comuna_key')) else 'N/A'
            ing_p = f'${row["ingresos_promedio"]/1e6:.1f}M' if pd.notna(row.get('ingresos_promedio')) and row['ingresos_promedio'] >= 1e6 else '$0'
            ing_t = f'${row["ingresos_total"]/1e9:.2f}B' if pd.notna(row.get('ingresos_total')) and abs(row['ingresos_total']) >= 1e9 else (f'${row["ingresos_total"]/1e6:.0f}M' if pd.notna(row.get('ingresos_total')) else 'N/A')
            dens = f'{row["densidad_empresarial"]:.1f}' if pd.notna(row.get('densidad_empresarial')) else 'N/A'
            area = f'{row["area_ha"]:.0f}' if pd.notna(row.get('area_ha')) else 'N/A'
            f.write(f"{comuna:<12} {int(row['total_empresas']):>9,} {ing_p:>10} {ing_t:>12} {int(row['empleo_total']):>8,} {row['pct_micro']:>7.1f}% {int(row['n_ciiu_distintos']):>5} {dens:>7} {area:>9}\n")
        
        f.write('-' * 100 + '\n')
        tot = gdf_merged['total_empresas'].sum()
        tot_e = gdf_merged['empleo_total'].sum()
        f.write(f"TOTAL MAPEADO: {int(tot):,} empresas | {int(tot_e):,} empleos\n\n")
    
    # 4. Top sectores
    f.write('-' * 100 + '\n')
    f.write('4. TOP 10 SECTORES ECONOMICOS\n')
    f.write('-' * 100 + '\n')
    col_sector = [c for c in df.columns if 'sector' in c or 'nombre_sector' in c]
    if col_sector:
        top_sect = df[col_sector[0]].value_counts().head(10)
        for sector, cuenta in top_sect.items():
            f.write(f"  {sector}: {cuenta:,} ({cuenta/len(df)*100:.1f}%)\n")
    f.write('\n')
    
    # 5. Top CIIU
    f.write('-' * 100 + '\n')
    f.write('5. TOP 15 ACTIVIDADES ECONOMICAS (CIIU)\n')
    f.write('-' * 100 + '\n')
    col_desc_ciiu = [c for c in df.columns if 'desc' in c and 'ciiu' in c]
    if col_desc_ciiu:
        top_ciiu = df[col_desc_ciiu[0]].value_counts().head(15)
        for act, cuenta in top_ciiu.items():
            f.write(f"  {act[:70]}: {cuenta:,} ({cuenta/len(df)*100:.1f}%)\n")
    f.write('\n')
    
    # 6. Resumen estadistico
    f.write('-' * 100 + '\n')
    f.write('6. RESUMEN ESTADISTICO DE INGRESOS OPERACIONALES\n')
    f.write('-' * 100 + '\n')
    ing_data = df[col_ingresos].dropna()
    f.write(f'  Registros con dato: {len(ing_data):,} ({len(ing_data)/len(df)*100:.1f}%)\n')
    f.write(f'  Media:    ${ing_data.mean()/1e6:.1f}M\n')
    f.write(f'  Mediana:  ${ing_data.median()/1e6:.1f}M\n')
    f.write(f'  P25:      ${ing_data.quantile(0.25)/1e6:.1f}M\n')
    f.write(f'  P75:      ${ing_data.quantile(0.75)/1e6:.1f}M\n')
    f.write(f'  Maximo:   ${ing_data.max()/1e9:.2f}B\n')
    f.write(f'  Con ingresos > 0: {(ing_data > 0).sum():,}\n')
    f.write(f'  Con ingresos = 0: {(ing_data == 0).sum():,}\n\n')
    
    # 7. Resumen empleo
    f.write('-' * 100 + '\n')
    f.write('7. RESUMEN ESTADISTICO DE EMPLEO\n')
    f.write('-' * 100 + '\n')
    emp_data = df[col_empleo].dropna()
    f.write(f'  Registros con dato: {len(emp_data):,}\n')
    f.write(f'  Media:    {emp_data.mean():.1f} empleados\n')
    f.write(f'  Mediana:  {emp_data.median():.0f} empleados\n')
    f.write(f'  Maximo:   {emp_data.max():,.0f} empleados\n')
    f.write(f'  Total:    {emp_data.sum():,.0f} empleados\n\n')
    
    # Cierre
    f.write('=' * 100 + '\n')
    f.write('FIN DEL REPORTE CONSOLIDADO\n')
    f.write('=' * 100 + '\n')

print(f'[OK] Reporte guardado: {REPORTE_PATH}')
print(f'     Tamano: {REPORTE_PATH.stat().st_size / 1024:.1f} KB')

## 13. Notas

- **Entorno local:** Usar `uv run` o activar el ambiente conda. Dependencias: pandas, openpyxl, geopandas, folium, matplotlib
- **Entorno Colab:** Descomentar la celda de instalacion de paquetes. El repo se clona automaticamente.
- **GeoJSON:** Ubicado en `data/info_geo/geojson_comunas/Comunas.geojson` (22 comunas de Cali)
- **Match de comunas:** El GeoJSON usa 'Comuna 6' y el Registro Mercantil 'Comuna 06'. Se normaliza con zero-padding.
- **Comunas faltantes:** El GeoJSON tiene 22 comunas, el Registro Mercantil tiene 39 zonas (incluye corregimientos).
- **Salidas:** PNG en `outputs/`, HTML interactivo en `outputs/`, reporte `outputs/consolidado.txt`
- **Reporte consolidado.txt:** Se genera automaticamente con tablas de indicadores, sectores, CIIU y estadisticas.